# NB05 — Phase 3d: price the alternative recipe


"This is where the "so what" gets teeth, and where the practical result lives" - Claude

This notebook deconfounds frame recovery and selfhood in the frozen arm training by giving just the coordinate frame to a non-self model.
NB03 is able to show that activation alignment is necessary while this tests whether it is sufficient.

`P-ridge-frozen` arm tests if the projector is recoverable at all by using a regression.
NB03 is only able to test if it is recoverable at certain N and random init.
`P-rand` vs `P-ridge` separates non-self target model from poorly initialized projector.

These results clarify evidence from the paper.

Qwen3-8B was only ever run with a randomly initialized projector in the paper. 
The release says otherwise: `config/feature_descriptions/qwen_131k.yaml` sets `use_embed_proj: true` and
loads `alignment_outputs/qwen_llama_3.1_8b_base/final_alignment_model.pt`, writing to a checkpoint
directory named `aligned_qwen_pretrained`.
We know that the random-init has a separate config, `qwen_131k_nonpretrainedembed.yaml`, writing to `nonaligned_qwen`. 
Three things follow.

**(a) The paper's description of its own method is inaccurate.** 
3.1 states that pretrained projections were included only for Llama-3.1-70B, but the released code has evidence that a pretrained Qwen projector was built and used.

**(b) The papers easiest test of basis compatibility was likely run or reported.**
`aligned_qwen_pretrained` vs `nonaligned_qwen` is same explainer, target, and data, with
alignment as the only variable. 
Table 1's Qwen row may also not be internally consistent about which condition it reflects: only the LM-judge config carries the
pretrained path, while `_simcor`, `_ood_fw` and `_ood_diff` do not, and `peft_lora` differs across them.

**(c) The new hypothesis, which is cheap to test and sharper than anything in v2.** 
Because projectors are LoRA target modules at rank 128, a randomly initialized 4096 × 4096 projector is essentially unfixable in post training.
A pretrained one only needs refinement, so it should be fixable.
THe papers claim that "activation alignment improves explainer performance," may substantially be:

> **Under low-rank training, projector initialization dominates, and the random-vs-pretrained gap
> is largely a rank-128 artifact rather than evidence about activation alignment.**

We test ths below with one extra arm: `P-rand-full` is `P-rand` with the rank cap lifted and nothing else changed.

In the paper Llama-3.1-70B with a pretrained projection recovers 14% on SAE explanations, 2.7x on real activations, and 1.7x on activation differences relative to random init.
Meanwhile Qwen3-8B, the main cross-model comparison in Table 1, was only ever run with a random init projector, so the headline cross-model gap and the projector-quality effect are confounded.

The `P-ridge-frozen` arm is a closed-form ridge fit evaluated with an explainer that is not fine tuned on this target.
Training a cross-model explainer from ridge init is still a per-target training run but shows you only need to regress a linear map between activations.

There is also an analytic result here which we check numerically below.
Fitting ridge against rotated activations gives exactly `Π Q^T`.
So theoretically, the closed form recipe is as good on rotated activations as unrotated ones.  


In [ ]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


In [ ]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# The ridge fit holds the 8B target and the 4B cross-model resident together.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="required", min_vram_gb=80)

import se_config as C


In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch

import rotate as R
import se_common as S

CROSS_MODEL_ID = "Qwen/Qwen3-4B"      # primary; 1.7B is the secondary in v2 §3

tokenizer = S.load_tokenizer()
Q, _ = R.load_rotation(f"{C.ROTATION_DIR}/Q_seed{C.Q_SEED}.pt")

with open(f"{C.ROTATION_DIR}/gate_texts.json") as f:
    texts = json.load(f)              # the same FineWeb sample NB01 gated on
print(f"{len(texts)} texts for the paired-activation fit")


## 1. Load both models, and fix the layer correspondence

`chunk_hidden_states` splits each model's layers into `N_LAYER_CHUNKS` groups and averages within a group. 
Applied to both models independently chunk `k` of the 4B model covers the same fractional depth range as chunk `k` of the 8B model.

A shared tokenizer is required for token alignment, which holds within Qwen3.


In [ ]:
from transformers import AutoModelForCausalLM

dtype = S.compute_dtype()
target_model = AutoModelForCausalLM.from_pretrained(
    C.TARGET_MODEL_ID, dtype=dtype, device_map="auto").eval()
cross_model = AutoModelForCausalLM.from_pretrained(
    CROSS_MODEL_ID, dtype=dtype, device_map="auto").eval()

d_M, d_E = target_model.config.hidden_size, cross_model.config.hidden_size
L_M, L_E = target_model.config.num_hidden_layers, cross_model.config.num_hidden_layers
print(f"target {C.TARGET_MODEL_ID}: d = {d_M}, layers = {L_M}")
print(f"cross  {CROSS_MODEL_ID}: d = {d_E}, layers = {L_E}")

assert C.LAYER_CORRESPONDENCE == "proportional_depth"
print(f"\nlayer correspondence rule: {C.LAYER_CORRESPONDENCE}, via {C.N_LAYER_CHUNKS} "
      f"contiguous chunks per model")
for name, L in (("target", L_M), ("cross", L_E)):
    base, rem = divmod(L, C.N_LAYER_CHUNKS)
    sizes = [base + (1 if c < rem else 0) for c in range(C.N_LAYER_CHUNKS)]
    bounds, start = [], 0
    for s in sizes:
        bounds.append(f"{start}-{start + s - 1}")
        start += s
    print(f"  {name:>6}: {bounds}")

assert tokenizer.vocab == S.load_tokenizer(CROSS_MODEL_ID).vocab, \
    "token alignment requires a shared vocabulary"
print("\nshared vocabulary confirmed")


## 2. Alignment, before and after rotation

Measuring the 3.4 / Table 4 quantity to validate the paper.
For the self-explainer the identity arm alignment is 1 and the rotated arm alignment is chance.


In [ ]:
@torch.no_grad()
def coordinate_alignment(model, tokenizer, texts, transform=None, n_chunks=C.N_LAYER_CHUNKS,
                         batch_size=2, max_length=256, n_batches=8):
    """Mean cosine between a model's chunk-averaged activation and its transformed self."""
    sims = []
    for i, start in enumerate(range(0, len(texts), batch_size)):
        if i >= n_batches:
            break
        enc = tokenizer(texts[start:start + batch_size], return_tensors="pt", padding=True,
                        truncation=True, max_length=max_length)
        mask = enc["attention_mask"].bool()
        out = model(**{k: v.to(model.device) for k, v in enc.items()}, output_hidden_states=True)
        for chunk in S.chunk_hidden_states(out.hidden_states, n_chunks):
            h = chunk.cpu()[mask].double()
            h2 = h if transform is None else h @ transform.double().T
            sims.append(torch.nn.functional.cosine_similarity(h, h2, dim=1).mean().item())
    return float(np.mean(sims))


align_identity = coordinate_alignment(target_model, tokenizer, texts)
align_rotated = coordinate_alignment(target_model, tokenizer, texts, transform=Q)
print(f"self-alignment, R-id : {align_identity:+.4f}")
print(f"self-alignment, R-Q  : {align_rotated:+.4f}   (chance scale ~{1/np.sqrt(d_M):.4f})")
print("\nThe rotation sends the paper's alignment quantity to chance while leaving the model")
print("that produced the activations bit-for-bit identical in function.")

with open(f"{C.REPORTS_DIR}/alignment.json", "w") as f:
    json.dump({"identity": align_identity, "rotated": align_rotated,
               "chance_scale": float(1 / np.sqrt(d_M))}, f, indent=2)


## 3. Fit `Π_ℓ` in closed form

$$\Pi_\ell = H^E (H^M)^\top \left(H^M (H^M)^\top + \lambda I\right)^{-1}$$

We run ridge regression.

The two self-explainer fits have known answers (`I` and `Q^T`), which we check before the code is used for anything.


In [ ]:
FIT_TEXTS = texts[:128]
HOLDOUT = texts[128:192]
LAMBDA = 1e-2

fits = {}
for name, explainer, transform in [
    ("self_identity", target_model, None),
    ("self_rotated", target_model, Q),
    ("cross_identity", cross_model, None),
    ("cross_rotated", cross_model, Q),
]:
    t0 = time.time()
    mats, info = S.fit_ridge_projector(explainer, target_model, tokenizer, FIT_TEXTS,
                                       lam=LAMBDA, vector_transform=transform)
    resid = S.ridge_residual(mats, explainer, target_model, tokenizer, HOLDOUT,
                            vector_transform=transform)
    fits[name] = mats
    print(f"{name:>16}: {info['n_tokens']} tokens, {time.time()-t0:.0f}s, "
          f"held-out residual per chunk = {[round(r, 3) for r in resid]}")

torch.save({k: [m.float() for m in v] for k, v in fits.items()},
           f"{C.ROTATION_DIR}/ridge_projectors.pt")
print(f"\nsaved to {C.ROTATION_DIR}/ridge_projectors.pt")

# Appendix F.4: no alignment-training code ships with the release — only the loading path,
# which consumes LinearAlignmentModule state dicts keyed alignments.{layer}.weight
# (model/utils.py:121-156), and the .pt artifacts are not public. Emitting that shape costs
# nothing and makes this fit drop into their pipeline unmodified, which is the difference
# between a result and a reusable artifact.
for name in ("cross_identity", "cross_rotated"):
    info = S.save_alignment_state_dict(
        fits[name], f"{C.ROTATION_DIR}/alignment_{name}.pt", n_layers=L_E)
    print(f"  {name}: {info['n_layers']} layer entries -> {info['path']}")
print("keyed alignments.{layer}.weight, one entry per explainer layer, chunk matrices "
      "broadcast\nunder the proportional-depth rule above")


In [ ]:
# Diagnostics with known answers.
print("self_identity should be I:")
for c, m in enumerate(fits["self_identity"]):
    st = R.compare_to_inverse(m.double(), torch.eye(d_M, dtype=torch.float64))
    print(f"  chunk {c}: rel_frob {st['rel_frobenius']:.4f}, row cos {st['row_cosine_mean']:+.4f}")

print("\nself_rotated should be Q^T:")
for c, m in enumerate(fits["self_rotated"]):
    st = R.compare_to_inverse(m.double(), Q)
    print(f"  chunk {c}: rel_frob {st['rel_frobenius']:.4f}, row cos {st['row_cosine_mean']:+.4f}, "
          f"orthogonality defect {st['orthogonality_defect']:.4f}")

print("\nRidge shrinkage keeps rel_frob slightly above zero. Large values mean the fit is")
print("broken and nothing below is interpretable.")


### Analytic result

Fitting the ridge map against rotated activations gives *exactly* `Π Q^T`:

$$\Pi^{Q} = H^E (QH^M)^\top\!\left(QH^M H^{M\top}\!Q^\top + \lambda I\right)^{-1}
= H^E H^{M\top}\!\left(H^M H^{M\top} + \lambda I\right)^{-1}\!Q^\top = \Pi Q^\top$$

`se/test_input_map.py` proves this on synthetic data. 
Here it is checked on the real fits, where the only sources of disagreement are bf16 activations and the streaming
accumulation order.

**The implication is the strong version of §1's practical claim.** 
The closed-form recipe is theoretically equally as good on rotated and rotated activations.
This means if the ridge regression closes the gap, then the coordinate frame is not the mechanism for self-explainer superiority.


In [ ]:
print("checking Pi^Q == Pi Q^T on the fitted maps\n")
rows = []
for pair, name in [(("self_identity", "self_rotated"), "self"),
                   (("cross_identity", "cross_rotated"), "cross")]:
    for c, (pi, pi_q) in enumerate(zip(fits[pair[0]], fits[pair[1]])):
        expected = pi.double() @ Q.T
        rel = ((pi_q.double() - expected).norm() / expected.norm()).item()
        rows.append({"pair": name, "chunk": c, "relative_error": rel})
        print(f"  {name:>6} chunk {c}: ||Pi^Q - Pi Q^T||_F / ||Pi Q^T||_F = {rel:.2e}")

analytic = pd.DataFrame(rows)
with open(f"{C.REPORTS_DIR}/ridge_rotation_identity.json", "w") as f:
    json.dump({"max_relative_error": float(analytic.relative_error.max()),
               "per_chunk": rows}, f, indent=2)
print(f"\nmax relative error: {analytic.relative_error.max():.2e}")
print("bf16 activations carry ~3 decimal digits, so ~1e-3 is the arithmetic. Larger means")
print("the two fits saw different data — check that vector_transform is the only difference.")


## 4. `P-ridge-frozen` arm

We do a closed-form fit and then immediately evaluate it. 
If this approaches `E_self`, the practical conclusion is not "a good initialization speeds things up" but "A self-explainer per target is unnecessary and one good explainer and a ridge regression will suffice."*

Evaluated under both rotations.


In [ ]:
ridge = torch.load(f"{C.ROTATION_DIR}/ridge_projectors.pt", weights_only=True)

del target_model, cross_model
import gc
gc.collect()
torch.cuda.empty_cache()

datasets = {arm: S.load_ready_dataset(arm) for arm in ("identity", "Q")}

frozen_results = []
for label, rot, key in [
    ("P-ridge-frozen · R-id", "identity", "cross_identity"),
    ("P-ridge-frozen · R-Q", "Q", "cross_rotated"),
]:
    print(f"\n=== {label} (no training at all) " + "=" * 22)
    scores = S.eval_run(None, rot, "C0", "ridge", tokenizer, dataset=datasets[rot],
                        ridge_matrices=ridge[key], explainer_model_id=CROSS_MODEL_ID)
    scores["label"] = label
    frozen_results.append(scores)
    print(f"  exact_match {scores['exact_match']:.3f} | "
          f"has_changed_f1 {scores['has_changed_f1']:.3f} | "
          f"content_match {scores['content_match']:.3f}")

frozen = pd.DataFrame(frozen_results)
print()
print(frozen[["label", "exact_match", "has_changed_f1", "content_match"]].round(3).to_string(index=False))
print("\nThese two should agree: the analytic result says the frozen ridge map is exactly as")
print("good on rotated activations as on unrotated ones.")


### A caveat on the frozen arm

An explainer that was never trained on this task will mostly fail to format its answer, so
`exact_match` may be near zero for reasons that have nothing to do with the projector. 
We read `has_changed_f1` and `content_match` here to  compare against the untrained baseline from
NB03 rather than against the trained self-explainer alone. 

If it does not clear the untrained baseline, the honest report is that the frozen recipe fails at this scale and the trained-from-ridge arm below is the recoverable version of the claim.


## 5. Trained arms and the rank-artifact test

Four arms, and the first pair is hypothesis (c) from the header:

| Arm | Input map | What it isolates |
|---|---|---|
| `P-rand` | random dense, **rank-128 update only** | the paper's cross-model condition, faithfully |
| `P-rand-full` | random dense, full-rank trainable | the same handicap without the rank cap |
| `P-ridge` | closed-form init, then trained | what a good initialization is worth |
| ~~`P-ridge` · R-Q~~ | the same on rotated activations | **off by default** — see below |

`P-rand` and `P-rand-full` differ only in whether the projector can move off its random initialization by more than rank 128. 
If `P-rand-full` closes most of the distance to `P-ridge`, then "pretrained projectors beat random ones" is substantially a statement about rank capacity rather than about activation alignment. 
If it does not, rank is not the binding constraint and the paper's reading stands. 
The preregistration has a row for it either way (`rank_artifact` / `rank_not_binding`).

**Budget.** The full grid is 29 missing runs, ~43.5 h. The cell below runs 13 of them
(~19.5 h), because that is what the readings consume:

- **`P-ridge · R-Q` is off.** §3's analytic result already establishes `Π^Q = Π Qᵀ` three
  ways — proved on synthetic data in `se/test_input_map.py`, verified numerically on the
  real fitted maps in §3's last cell, and checked end-to-end for free by the two
  `P-ridge-frozen` arms in §4. Training it is a fourth confirmation, and no reading in NB05
  or NB07 reads it: it appears only as a bar in NB07's Figure 3. Set `RUN_RIDGE_RQ = True`
  to buy it back.
- **The closure test runs at N = 512 and 2,048, not 16,384.** The hypothesis is that a
  rank-128 cap leaves a random projector unfixable while a pretrained one only needs
  refinement — which bites hardest where training data is scarce. At 16,384 a rank-128
  update has ample signal to fix a random init, so `P-rand → P-ridge` is the gap most
  likely to close to nothing and hand `closure_with_ci` an unresolvable denominator. That
  is 16.5 h to be told the gap vanished.
- **One `P-ridge · R-id` point survives at 16,384, single-seed**, for the practical claim
  NB07 §18 leads with (ridge recipe against `E_self` at the largest N). The closure test
  does not read it, so it needs no band.


In [ ]:
# the (capacity, init) pairs live in se_config so NB03, NB05 and NB07 cannot disagree about
# what "P-rand" means — the distinction between P-rand and P-rand-full is the whole test
RAND_CAP, RAND_INIT = C.PROJECTOR_ARMS["P-rand"]            # C128 / random: the rank cap
FULL_CAP, FULL_INIT = C.PROJECTOR_ARMS["P-rand-full"]       # Cfull / random: cap lifted
print(f"P-rand      = {RAND_CAP}/{RAND_INIT}  (rank {C.CAPACITY_RANK[RAND_CAP]} update)")
print(f"P-rand-full = {FULL_CAP}/{FULL_INIT}  (full-rank trainable)")

# --- budget -----------------------------------------------------------------------------
# The full grid (every arm x N_TRAIN_LADDER x 3 seeds) is 29 missing runs, ~43.5 h. What the
# readings actually consume is 13 of them. Restore any line below to buy the rest back.
#
# CLOSURE_N -- where the rank-artifact test runs, all three arms at three seeds. The
# hypothesis is that a rank-128 cap makes a random projector unfixable while a pretrained
# one only needs refinement, so it bites hardest where data is scarce. At 16,384 a rank-128
# update has ample signal to fix a random init, which is precisely the case where
# P-rand -> P-ridge closes to nothing and hands `closure_with_ci` an unresolvable
# denominator -- 16.5 h to be told the gap vanished.
CLOSURE_N = [512, 2048]

# RECIPE_N -- one un-replicated P-ridge point at the top N, for the practical claim NB07 §18
# leads with (ridge recipe vs E_self at the largest N). The closure test does not read it, so
# a single seed is enough.
RECIPE_N = [max(C.N_TRAIN_LADDER)]

# P-ridge · R-Q is OFF. Cell 11 proves Pi^Q = Pi Q^T analytically, se/test_input_map.py
# proves it on synthetic data, cell 12 verifies it numerically on the real fitted maps, and
# cell 14 checks it end-to-end for free on the two frozen arms. Training it is a fourth
# confirmation of an identity already established three ways, and no reading in NB05 or NB07
# consumes it -- it appears only as a bar in NB07's Figure 3. If it ever disagreed, the
# explanation would be SGD noise rather than a basis effect, because the identity holds.
# First thing to add back if pod time turns out to be cheap.
RUN_RIDGE_RQ = False


def seeds_for_recipe(n):
    """Three seeds where the closure test needs a spread; one where it does not.

    Deliberately NOT C.seeds_for_arm -- that encodes which self-explainer arms were
    replicated at 16,384 and would hand P-ridge three seeds while handing P-rand one,
    breaking §5's closure check outright.
    """
    return C.seeds_for(n) if n in CLOSURE_N else [C.SEED]


jobs = [
    # (label,             rotation,   capacity,  init,       ridge key,        N values)
    ("P-rand",            "identity", RAND_CAP,  RAND_INIT,  None,             CLOSURE_N),
    ("P-rand-full",       "identity", FULL_CAP,  FULL_INIT,  None,             CLOSURE_N),
    ("P-ridge · R-id",    "identity", "Cfull",   "ridge",    "cross_identity", CLOSURE_N + RECIPE_N),
]
if RUN_RIDGE_RQ:
    jobs.append(
    ("P-ridge · R-Q",     "Q",        "Cfull",   "ridge",    "cross_rotated",  CLOSURE_N + RECIPE_N))

# Priced before anything starts. `seeds_for_recipe` above is the budget; every arm the
# closure test reads gets every seed at CLOSURE_N, which is what its gap CI needs.
PLAN = S.plan_runs([dict(n_train=n, rotation=rot, capacity=cap, init=init,
                         explainer_model_id=CROSS_MODEL_ID, seed=seed)
                    for label, rot, cap, init, key, n_values in jobs
                    for n in n_values for seed in seeds_for_recipe(n)],
                   label="NB05 §5 ridge recipe + rank-artifact arms")

recipe_results = list(frozen_results)
for label, rot, cap, init, key, n_values in jobs:
    for n in n_values:
        for seed in seeds_for_recipe(n):
            print(f"\n=== {label} · n={n} · seed={seed} " + "=" * 30)
            kw = dict(tokenizer=tokenizer, dataset=datasets[rot],
                      ridge_matrices=ridge[key] if key else None,
                      explainer_model_id=CROSS_MODEL_ID, seed=seed)
            S.run_training(n, rot, cap, init, **kw)
            scores = S.eval_run(n, rot, cap, init, **kw)
            scores["label"] = label
            recipe_results.append(scores)
            print(f"  exact_match {scores['exact_match']:.3f} | "
                  f"content_match {scores['content_match']:.3f}")

recipe = pd.DataFrame(recipe_results)
recipe.to_csv(f"{C.REPORTS_DIR}/ridge_recipe.csv", index=False)
recipe[["label", "explainer", "rotation", "capacity", "init", "n_train",
        "exact_match", "has_changed_f1", "content_match"]].round(3)

### Reading the rank-artifact test

The quantity is what fraction of the `P-rand → P-ridge` gap lifting the rank cap recovers:

$$\text{closure} = \frac{\text{score}(\texttt{P-rand-full}) - \text{score}(\texttt{P-rand})}
{\text{score}(\texttt{P-ridge}) - \text{score}(\texttt{P-rand})}$$

Preregistered at `rank_artifact_closure = 0.50`.
Above it, the paper's random-vs-pretrained gap is largely a rank artifact and its mechanism needs restating.
Near zero, rank is not the binding constraint and initialization quality impacts alignment.

**The denominator is checked before the ratio is reported.** `P-ridge − P-rand` is a difference
of two measured arms, so it carries seed noise, and a closure computed against a gap that is
itself indistinguishable from zero is not a number — it is a division by noise. This is the same
defect that put `retained` values of +2.59 and −13.67 into the NB03/NB04 tables before
`fraction_retained` moved to a constant denominator (`se_config.CONTRIBUTION_SPAN`). Here no
constant is available, because the gap *is* the quantity of interest, so the cell below instead:

1. reports every arm's per-seed **sd and band**, including `P-ridge` — whose spread the previous
   version discarded outright (`ridge_v, _ = arm_score(...)`);
2. reports the gap with its standard error and a **t-based 95% CI**;
3. reports `closure` **only when that CI excludes zero**, with a **Fieller** interval rather than
   a point estimate — numerator and denominator share `P-rand`, so they are correlated, and
   `se_common.closure_with_ci` carries that covariance;
4. when the gap is not resolvable, prints the raw `P-rand-full − P-rand` difference instead and
   writes `closure` as null.

A closure whose Fieller CI straddles 0.50 decides nothing, and the cell says so rather than
picking the nearer side.


In [ ]:
prereg = json.load(open(f"{C.REPORTS_DIR}/preregistration.json"))
metric = prereg["primary_metric"]
closure_threshold = prereg["thresholds"]["rank_artifact_closure"]


def arm_seeds(label, n=None, metric=metric):
    """Every seed's score for one arm at one N — not the mean.

    The closure below divides by `P-ridge - P-rand`, so that gap's seed spread decides
    whether the ratio means anything. The previous version of this cell returned only a mean
    and a band, threw the band away for P-ridge (`ridge_v, _ = arm_score(...)`), and guarded
    the division with `if gap` — which is False only when the gap is exactly 0.0. A gap of
    0.001 against a seed band of 0.03 sailed through and produced a closure of 30.
    """
    sub = recipe[recipe.label == label]
    if n is not None and "n_train" in sub:
        sub = sub[sub.n_train == n]
    return [float(v) for v in sub[metric]] if len(sub) else []


def arm_score(label, n=None, metric=metric):
    """Mean and seed band, for printing. `arm_seeds` is what the statistics use."""
    xs = arm_seeds(label, n, metric)
    return (float(np.mean(xs)), float(max(xs) - min(xs))) if xs else (None, None)


print("THE RANK-ARTIFACT TEST (hypothesis (c))")
print("=" * 74)
rows = []
for n in C.N_TRAIN_LADDER:
    rand_s = arm_seeds("P-rand", n)
    full_s = arm_seeds("P-rand-full", n)
    ridge_s = arm_seeds("P-ridge · R-id", n)
    if not rand_s or not full_s:
        continue

    # Everything about the denominator lives here: its spread, its CI, and whether it is
    # separable from zero. `closure` comes back None when it is not.
    cl = S.closure_with_ci(rand_s, full_s, ridge_s)
    rows.append({"n_train": n,
                 "P-rand": cl["P-rand"]["mean"], "P-rand-full": cl["P-rand-full"]["mean"],
                 "P-ridge": cl["P-ridge"]["mean"],
                 "P-rand_sd": cl["P-rand"]["sd"], "P-rand-full_sd": cl["P-rand-full"]["sd"],
                 "P-ridge_sd": cl["P-ridge"]["sd"],
                 "gap": cl.get("gap"), "gap_se": cl.get("gap_se"),
                 "gap_ci95_lo": (cl.get("gap_ci95") or [None, None])[0],
                 "gap_ci95_hi": (cl.get("gap_ci95") or [None, None])[1],
                 "gap_resolvable": cl.get("gap_resolvable"),
                 "closure": cl.get("closure"),
                 "closure_ci95_lo": (cl.get("closure_ci95") or [None, None])[0],
                 "closure_ci95_hi": (cl.get("closure_ci95") or [None, None])[1],
                 "seed_band": cl.get("seed_band_max"),
                 "note": cl.get("note", "")})

    print(f"\nN = {n}")
    for lbl, key in [(f"P-rand      (rank {C.CAPACITY_RANK[RAND_CAP]} cap)", "P-rand"),
                     ("P-rand-full (no cap)     ", "P-rand-full"),
                     ("P-ridge     (good init)  ", "P-ridge")]:
        a = cl[key]
        if a["mean"] is None:
            print(f"  {lbl}: (missing)")
            continue
        print(f"  {lbl}: {a['mean']:.3f}"
              + (f"  +/- {a['sd']:.3f} sd, band {a['band']:.3f}" if a["sd"] is not None
                 else "  (1 seed — no spread)")
              + f"  [{a['n_seeds']} seed(s)]")

    # THE CHECK: is the denominator bigger than its own noise?
    if cl.get("gap") is not None and cl.get("gap_se") is not None:
        lo, hi = cl["gap_ci95"]
        print(f"  --> P-ridge - P-rand gap  : {cl['gap']:+.4f} +/- {cl['gap_se']:.4f} se, "
              f"t{cl['df']} 95% CI [{lo:+.4f}, {hi:+.4f}]")
        print(f"      denominator resolvable: {cl['gap_resolvable']} "
              f"({'CI excludes 0' if cl['gap_resolvable'] else 'CI INCLUDES 0'})")

    if cl.get("closure") is not None:
        ci = cl.get("closure_ci95")
        ci_txt = (f"  Fieller 95% CI [{ci[0]:+.2f}, {ci[1]:+.2f}]" if ci else
                  "  (no bounded CI — point estimate indicative only)")
        print(f"      closure of the rand->ridge gap: {cl['closure']:+.2f}{ci_txt}  "
              f"(threshold {closure_threshold})")
        if ci and ci[0] > closure_threshold:
            verdict = "rank artifact: the cap, not the alignment, was doing the work"
        elif ci and ci[1] < closure_threshold:
            verdict = "rank is not the binding constraint; the paper's reading stands"
        else:
            verdict = (f"undecided at this N — the CI straddles the {closure_threshold} "
                       f"threshold, so neither row is earned")
        print(f"      -> {verdict}")
    else:
        print(f"      closure: NOT REPORTED. {cl['note']}")
        print(f"      raw effect of lifting the cap: "
              f"{cl['P-rand-full']['mean'] - cl['P-rand']['mean']:+.4f} — report this instead.")

if rows:
    ra = pd.DataFrame(rows)
    ra.to_csv(f"{C.REPORTS_DIR}/rank_artifact.csv", index=False)
    with open(f"{C.REPORTS_DIR}/rank_artifact.json", "w") as f:
        json.dump({"threshold": closure_threshold, "rows": rows}, f, indent=2)
    print(f"\nwritten to {C.REPORTS_DIR}/rank_artifact.csv")

    unresolved = [r["n_train"] for r in rows if not r.get("gap_resolvable")]
    if unresolved:
        print(f"\n!! the P-ridge - P-rand gap is not separable from zero at N = {unresolved}.")
        print("   Closure is undefined there and is written as null, not as a number. If this")
        print("   is every N, the rank-artifact test did not run — it needs a real gap to")
        print("   normalize, and the finding to report is that P-rand and P-ridge did not")
        print("   separate, which is itself informative about hypothesis (c).")

print("\nNB07 applies the preregistered rule; do not classify the outcome here.")


Next: NB06 asks what the trained maps actually learned, and runs the control that must not
move.
